# Assessment 2: Machine Learning and Real-Time Streaming

**Student ID:** 35415169
**Unit:** ITO5202 – Data Processing for Big Data

## Part A: Machine Learning Pipeline

This assessment reuses the airline on-time performance dataset from Assessment 1. The dataset will first be divided into a historical training subset and a fully held-out streaming subset so that records replayed through Kafka in Part B have not been used during model training.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Assessment2_ML_Streaming")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.5


In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
) 

airline_schema = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY_OF_MONTH", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("FL_DATE", StringType(), True),
    StructField("OP_UNIQUE_CARRIER", StringType(), True),
    StructField("TAIL_NUM", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("CRS_DEP_TIME", IntegerType(), True),
    StructField("DEP_TIME", IntegerType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("DEP_DELAY_NEW", DoubleType(), True),
    StructField("DEP_DEL15", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", IntegerType(), True),
    StructField("ARR_TIME", IntegerType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("ARR_DELAY_NEW", DoubleType(), True),
    StructField("ARR_DEL15", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("FLIGHTS", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True)
])
    

In [3]:
file_path = "../T_ONTIME_REPORTING.csv"


df = (
    spark.read
    .option("header", True)
    .schema(airline_schema)
    .csv(file_path)
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


### Target preparation

The machine learning task is binary classification using 'ARR_DEL15' as the target variable. Records with a missing target are excluded because cancelled or diverted flights may not have a valid arrival-delay outcome.

In [5]:
ml_df = df.filter(F.col("ARR_DEL15").isNotNull())
print("Rows with valid ARR_DEL15 target:", ml_df.count())

Rows with valid ARR_DEL15 target: 522269


In [6]:
ml_df.groupBy("ARR_DEL15").count().orderBy("ARR_DEL15").show()

+---------+------+
|ARR_DEL15| count|
+---------+------+
|      0.0|424139|
|      1.0| 98130|
+---------+------+



### Train/stream split

A reproducible 70/30 split is applied using a fixed random seed of 42. The 70% subset is reserved for machine learning model development and evaluation, while the remaining 30% is held out entirely for the simulated real-time streaming stage in Part B.

In [7]:
train_df, stream_df = ml_df.randomSplit([0.7, 0.3], seed=42)

train_count = train_df.count()
stream_count = stream_df.count()

print("Training subset:", train_count)
print("Streaming subset:", stream_count)
print("Total:", train_count + stream_count)

Training subset: 365710
Streaming subset: 156559
Total: 522269


In [8]:
total_count = train_count + stream_count

train_pct = train_count / total_count * 100
stream_pct = stream_count / total_count * 100

print("Training percentage:", round(train_pct, 2), "%")
print("Streaming percentage:", round(stream_pct, 2), "%")

Training percentage: 70.02 %
Streaming percentage: 29.98 %


In [9]:
stream_output_path = "data/stream_data.parquet"

stream_df.write.mode("overwrite").parquet(stream_output_path)

print("streaming subset saved to:", stream_output_path)

streaming subset saved to: data/stream_data.parquet


In [10]:
stream_check = spark.read.parquet(stream_output_path)

print("Reloaded streaming rows:", stream_check.count())

Reloaded streaming rows: 156559
